### General Requirements 

In [32]:
%pip install beautifulsoup4
%pip install lxml
%pip install spacy
%pip install ipywidgets
%pip install transformers
%pip install nlp

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Prepare the dataset

### Open the original dataset

In [2]:
from bs4 import BeautifulSoup


# Reading the data inside the xml
# file to a variable under the name
# data
with open('deid_surrogate_train_all_version2.xml', 'r') as f:
    data = f.read()

# Passing the stored data inside
# the beautifulsoup parser, storing
# the returned object
Bs_data = BeautifulSoup(data, "xml")

# Using find() to extract attributes
# of the first instance of the tag
b_type = Bs_data.find_all('PHI', {'TYPE':'HOSPITAL'})

print(b_type)

[<PHI TYPE="HOSPITAL">FIH</PHI>, <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel
            Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Em Nysonken Medical Center</PHI>, <PHI TYPE="HOSPITAL">OLH</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Staviewordna University Of Medical Center</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">6U-489</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Hoseocon Medical Center</PHI>, <PHI TYPE="HOSPITAL">Heaonboburg Linpack Grant Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical
            Center</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">1D-419</PHI>, <PHI TYPE="HOSPITAL"

### Prepare as text

In [3]:
from spacy import displacy
import re

xml_text = Bs_data.get_text()
record_test = Bs_data.find('ROOT')

### Remove IDs if needed

In [4]:
def remove_ids(soup):
    for item in soup.find_all(attrs={"TYPE": "ID"}):
        item.string = "*****"
    return soup

### Remove IDs and labels

In [5]:
record_test = remove_ids(record_test) # Remove IDs from the XML data
record_text = record_test.get_text()
record_str = str(record_text)
print(record_str)




*****
FIH
*****
*****
*****
11/19/1994
            12:00:00 AM Discharge Summary Unsigned DIS Report Status : Unsigned ADMISSION DATE : 11/19/94 DISCHARGE DATE : 11/28/94
            ADMISSION DIAGNOSIS : Aspiration pneumonia , esophageal laceration . HISTORY OF PRESENT
            ILLNESS : Mr. Blind is a 79-year-old white white male with a
            history of diabetes mellitus , inferior myocardial infarction , who underwent open
            repair of his increased diverticulum November 13th at Sephsandpot Center . The patient developed hematemesis November 15th and was intubated for respiratory distress . He was
            transferred to the Valtawnprinceel Community Memorial Hospital
            for endoscopy and esophagoscopy on the 16th of November which
            showed a 2 cm linear tear of the esophagus at 30 to 32 cm . The patient 's
            hematocrit was stable and he was given no further intervention . The patient attempted a
            gastrografin swallow o

### Remove IDs but keep labels

In [6]:
record_test = remove_ids(record_test) # Remove IDs from the XML data
record_str = str(record_test)
print(record_str)

<RECORD ID="640">
<TEXT>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="HOSPITAL">FIH</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="DATE">11/19</PHI>/1994
            12:00:00 AM Discharge Summary Unsigned DIS Report Status : Unsigned ADMISSION DATE : <PHI TYPE="DATE">11/19</PHI>/94 DISCHARGE DATE : <PHI TYPE="DATE">11/28</PHI>/94
            ADMISSION DIAGNOSIS : Aspiration pneumonia , esophageal laceration . HISTORY OF PRESENT
            ILLNESS : Mr. <PHI TYPE="PATIENT">Blind</PHI> is a 79-year-old white white male with a
            history of diabetes mellitus , inferior myocardial infarction , who underwent open
            repair of his increased diverticulum <PHI TYPE="DATE">November 13th</PHI> at <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI> . The patient developed hematemesis <PHI TYPE="DATE">November 15th</PHI> and was intubated for respiratory distress . He was
            transferred to the <PHI TYPE="HOSPITAL">Valtawnprinceel Co

### Retransform to xml file

In [38]:
import xml.etree.ElementTree as ET

# Parse the XML string
root = ET.fromstring(record_str)

# Create an ElementTree object
tree = ET.ElementTree(root)

# Write the ElementTree object to an XML file
tree.write("deid_without_ids.xml", encoding="utf-8", xml_declaration=True)

# Print the content of the XML file
with open("deid_without_ids.xml", "r") as f:
    data = f.read()
print(data)

ParseError: not well-formed (invalid token): line 3, column 0 (<string>)

# Bio NER

### Load the specialized model

In [6]:
from transformers import AutoModelForTokenClassification

bio_ner_model = AutoModelForTokenClassification.from_pretrained("blaze999/Medical-NER")

### Load the tokenizer

In [28]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("blaze999/Medical-NER")

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

### Create an instance of pipeline with the model and the tokenizer

In [29]:
from transformers import pipeline

ner_pipe = pipeline("ner", model=bio_ner_model, tokenizer=tokenizer)

Device set to use cpu


### Extract entities

In [1]:
ner_results = ner_pipe(record_str)
print(ner_results)

NameError: name 'ner_pipe' is not defined

### Mask choosen entities

In [49]:
# Define the entities to mask
labels_to_mask = ["I-AREA", "B-AREA",
                     "B-DATE", "I-DATE",
                       "I-NONBIOLOGICAL_LOCATION", "B-NONBIOLOGICAL_LOCATION",
                         "I-OCCUPATION", "B-OCCUPATION",
                         "I-PERSONAL_BACKGROUND", "B-PERSONAL_BACKGROUND",
                         ]
pattern_date = re.compile("[0-9]{2}\/[0-9]{2}\/[0-9]{2,4}")

entities_to_mask = []
entities_to_keep = []

def find_entities_to_mask(text, labels_to_mask, entities_to_mask, entities_to_keep):
    for ent in text:
        if  ent['entity'] in labels_to_mask:
            entities_to_mask.append(ent)
        else:
            entities_to_keep.append(ent)

find_entities_to_mask(ner_results, labels_to_mask, entities_to_mask, entities_to_keep)

print("entities to mask : ", entities_to_mask)
print("----------------------------------")
print("entities to keep : ", entities_to_keep)

# # Fonction pour masquer les entités
# def mask_entities(text, labels_to_mask):
#     for ent in text:
#         if ent.label_ in entities_to_mask:
#             masked_text = masked_text.replace(ent.word, "*****")
#         if pattern_date.match(ent.word):
#             masked_text = masked_text.replace(ent.word, "DATE")
#     return masked_text

# # Mask the entities
# masked_text = mask_entities(ner_results, entities_to_mask)

# # Print the masked text
# print(masked_text)

# masked_xml = nlp(masked_text)
#displacy.serve(masked_xml, style="ent")

entities to mask :  [{'entity': 'B-DATE', 'score': 0.56501704, 'index': 1197, 'word': '▁24', 'start': 6225, 'end': 6228}, {'entity': 'I-DATE', 'score': 0.45106906, 'index': 1198, 'word': 'th', 'start': 6228, 'end': 6230}, {'entity': 'B-NONBIOLOGICAL_LOCATION', 'score': 0.7063997, 'index': 1359, 'word': '▁Coronary', 'start': 7149, 'end': 7158}, {'entity': 'I-NONBIOLOGICAL_LOCATION', 'score': 0.8555178, 'index': 1360, 'word': '▁Care', 'start': 7158, 'end': 7163}, {'entity': 'I-NONBIOLOGICAL_LOCATION', 'score': 0.8959149, 'index': 1361, 'word': '▁Unit', 'start': 7163, 'end': 7168}, {'entity': 'B-DATE', 'score': 0.8010939, 'index': 1364, 'word': '▁25', 'start': 7175, 'end': 7178}, {'entity': 'I-DATE', 'score': 0.81900823, 'index': 1365, 'word': 'th', 'start': 7178, 'end': 7180}, {'entity': 'B-NONBIOLOGICAL_LOCATION', 'score': 0.5373869, 'index': 1417, 'word': '▁OL', 'start': 7461, 'end': 7464}, {'entity': 'I-NONBIOLOGICAL_LOCATION', 'score': 0.842804, 'index': 1418, 'word': 'H', 'start': 7